# Credit Card Fraud Detection: SVM vs Decision Tree

**Course:** IBM Machine Learning with Python

## Objective
Compare a `DecisionTreeClassifier` against a `LinearSVC` (Support Vector Machine) on a highly imbalanced credit card fraud dataset, using ROC-AUC as the evaluation metric. Then study how feature selection affects each model differently.

## Workflow
1. Import libraries
2. Load the dataset
3. Visualize class imbalance
4. Explore correlation with the target (`Class`)
5. Standardize and normalize features
6. Train/test split
7. Handle class imbalance with sample weights
8. Train and evaluate a Decision Tree
9. Train and evaluate a Linear SVM
10. Select the top 6 correlated features
11. Re-evaluate both models on the reduced feature set (Q3/Q4)
12. Draw conclusions (Q5)

## 1. Imports
- `StandardScaler`: standardizes features to zero mean / unit variance
- `compute_sample_weight`: computes per-sample weights to counter class imbalance
- `DecisionTreeClassifier`, `LinearSVC`: the two models being compared
- `roc_auc_score`: evaluation metric robust to class imbalance

In [ ]:
from __future__ import print_function
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.svm import LinearSVC

import warnings
warnings.filterwarnings('ignore')

## 2. Load the Dataset
Each row is a credit card transaction. `Class` is the target: `1` for fraud, `0` for legitimate. Features `V1`...`V28` are PCA-transformed for privacy, plus `Time` and `Amount`.

In [ ]:
# download the dataset
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/creditcard.csv"

# read the input data
raw_data = pd.read_csv(url)
raw_data

## 3. Visualize Class Imbalance
A pie chart of the `Class` value counts. Fraud detection datasets are typically extremely imbalanced (fraud is rare), which is why accuracy alone is a poor metric here — ROC-AUC is used instead.

In [ ]:
# get the set of distinct classes
labels = raw_data.Class.unique()

# get the count of each class
sizes = raw_data.Class.value_counts().values

# plot the class value counts
fig, ax = plt.subplots()
ax.pie(sizes, labels=labels, autopct='%1.3f%%')
ax.set_title('Target Variable Value Counts')
plt.show()

## 4. Correlation with the Target
A quick look at which features have the strongest linear relationship with `Class`.

In [ ]:
correlation_values = raw_data.corr()['Class'].drop('Class')
correlation_values.plot(kind='barh', figsize=(10, 6))

## 5. Standardize and Normalize Features
- `StandardScaler`: rescales columns `V1`...`Amount` to zero mean / unit variance (SVMs are sensitive to feature scale)
- `Time` is excluded from the feature matrix — it's a row index in time, not a predictive signal
- `normalize(..., norm="l1")`: additionally scales each row so its feature values sum (in absolute value) to 1

In [ ]:
# standardize features by removing the mean and scaling to unit variance
raw_data.iloc[:, 1:30] = StandardScaler().fit_transform(raw_data.iloc[:, 1:30])
data_matrix = raw_data.values

# X: feature matrix (for this analysis, we exclude the Time variable from the dataset)
X = data_matrix[:, 1:30]

# y: labels vector
y = data_matrix[:, 30]

# data normalization
X = normalize(X, norm="l1")

## 6. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## 7. Compute Sample Weights
Because fraud cases are rare, `compute_sample_weight('balanced', ...)` assigns higher weight to the minority class during training, so the model doesn't just learn to always predict "not fraud".

In [ ]:
w_train = compute_sample_weight('balanced', y_train)

## 8. Train the Decision Tree
`max_depth=4` keeps the tree shallow to reduce overfitting. `sample_weight=w_train` applies the class balancing computed above.

In [ ]:
# for reproducible output across multiple function calls, set random_state to a given integer value
dt = DecisionTreeClassifier(max_depth=4, random_state=35)

dt.fit(X_train, y_train, sample_weight=w_train)

## 9. Evaluate the Decision Tree
`predict_proba(...)[:, 1]` gives the predicted probability of the positive class (fraud), which `roc_auc_score` needs instead of hard 0/1 predictions.

In [ ]:
y_pred_dt = dt.predict_proba(X_test)[:, 1]

In [ ]:
roc_auc_dt = roc_auc_score(y_test, y_pred_dt)
print('Decision Tree ROC-AUC score : {0:.3f}'.format(roc_auc_dt))

## 10. Train the Linear SVM
`LinearSVC` with `class_weight='balanced'` applies the same idea as `compute_sample_weight` but built into the SVM itself, so it doesn't over-favor the majority class.

In [ ]:
svm = LinearSVC(class_weight='balanced', random_state=31, loss="hinge", fit_intercept=False)
svm.fit(X_train, y_train)

## 11. Evaluate the Linear SVM
`decision_function` returns a signed distance from the separating hyperplane, used directly as the score for `roc_auc_score` (SVC-based models don't need `predict_proba`).

In [ ]:
y_pred_svm = svm.decision_function(X_test)

In [ ]:
roc_auc_svm = roc_auc_score(y_test, y_pred_svm)
print("SVM ROC-AUC score: {0:.3f}".format(roc_auc_svm))

## 12. Identify the Top 6 Correlated Features
Sort features by absolute correlation with `Class` and keep the strongest 6.

In [ ]:
correlation_values = abs(raw_data.corr()['Class']).drop('Class')
correlation_values = correlation_values.sort_values(ascending=False)[:6]
correlation_values

## 13. Rebuild the Feature Matrix with Only the Top 6 Features
Column indices `[3, 10, 12, 14, 16, 17]` in `data_matrix` correspond to the top 6 correlated features identified above.

In [ ]:
X = data_matrix[:, [3, 10, 12, 14, 16, 17]]

## Q3. Decision Tree on the Reduced Feature Set
Retrain and re-evaluate the Decision Tree using only the top 6 features.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
w_train = compute_sample_weight('balanced', y_train)

dt = DecisionTreeClassifier(max_depth=4, random_state=35)
dt.fit(X_train, y_train, sample_weight=w_train)

y_pred_dt = dt.predict_proba(X_test)[:, 1]
roc_auc_dt = roc_auc_score(y_test, y_pred_dt)
print('Decision Tree ROC-AUC score (reduced features): {0:.3f}'.format(roc_auc_dt))

**Observation:** the ROC-AUC value increases for the Decision Tree with this reduced feature set — the model benefits from feature selection since it only has to consider the most informative splits.

## Q4. SVM on the Reduced Feature Set
Retrain and re-evaluate the Linear SVM using only the top 6 features.

In [ ]:
svm = LinearSVC(class_weight='balanced', random_state=31, loss="hinge", fit_intercept=False)
svm.fit(X_train, y_train)

y_pred_svm = svm.decision_function(X_test)
roc_auc_svm = roc_auc_score(y_test, y_pred_svm)
print("SVM ROC-AUC score (reduced features): {0:.3f}".format(roc_auc_svm))

**Observation:** the ROC-AUC value decreases for the SVM with this reduced feature set — fewer dimensions make it harder for the SVM to construct an effective separating hyperplane.

## Q5. Inferences: Decision Trees vs SVMs
- With a larger set of features, SVM performed relatively better in comparison to the Decision Tree.
- The Decision Tree benefited from feature selection and performed better with fewer, more informative features.
- SVMs may require higher feature dimensionality to construct an efficient decision hyperplane.

## Conclusion
On this imbalanced fraud dataset, both models achieve strong ROC-AUC using sample/class weighting. Feature selection affects the two model families in opposite directions: Decision Trees gain from a focused, low-dimensional feature set, while linear SVMs rely on higher dimensionality to separate the classes well.